# 🧠 EX63: Multi-Object Tracking (MOT)

Tracking assigns persistent IDs across frames via:
1. YOLO detects boxes at frame $t$
2. **Kalman Filter** predicts each track's position at $t+1$
3. **Hungarian algorithm** on IoU matrix links predictions → detections
4. Unmatched detections → new IDs; unmatched tracks → deleted

| Feature | BoT-SORT (default) | ByteTrack |
|---------|--------------------|-----------|
| Re-ID | ✅ appearance | ❌ motion only |
| Camera motion | ✅ CMC | ❌ |
| Speed | Moderate | Fast |
| Best for | Moving cameras | Dense crowd |

**Key:** Set `persist=True` in frame-by-frame loops — without it, tracker state resets every call.

## 🔗 Links
- [[EX60_Streaming_Inference_TH]] | [[EX62_Inference_Visualization_TH]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, os
import cv2, numpy as np, torch
import matplotlib.pyplot as plt
from solution import track_objects_in_video
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")

# 2-object synthetic tracking video
video_path = "tracking_demo.mp4"
out = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*"mp4v"), 10.0, (640,480))
traj = {0:[], 1:[]}
for i in range(20):
    frame = np.zeros((480,640,3), dtype=np.uint8)
    cx1 = 100 + i*18; cx2 = 540 - i*18
    cv2.circle(frame, (cx1,200), 40, (255,180,50), -1)
    cv2.circle(frame, (cx2,280), 40, (50,180,255), -1)
    cv2.putText(frame, f"F{i:02d}", (10,20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    traj[0].append(cx1); traj[1].append(cx2)
    out.write(frame)
out.release()

print("\n--- AUDIT & INSPECTION START ---")
track_ids = track_objects_in_video("yolo11n.pt", video_path, tracker_config="bytetrack.yaml")
print(f"  Unique track IDs: {track_ids}")
print(f"  Count: {len(track_ids)} distinct objects")
print("--- AUDIT & INSPECTION END ---")

fig, axes = plt.subplots(1,2,figsize=(12,4))
frames = list(range(20))
axes[0].plot(frames, traj[0], "r-o", label="Object A"); axes[0].plot(frames, traj[1], "b-o", label="Object B")
axes[0].set_xlabel("Frame"); axes[0].set_ylabel("X position"); axes[0].set_title("GT Trajectories")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].bar(range(len(track_ids)), [1]*len(track_ids), tick_label=[f"ID {t}" for t in track_ids], color="#9b59b6")
axes[1].set_title(f"Track IDs ({len(track_ids)} unique)"); axes[1].set_ylabel("Active")
plt.tight_layout(); plt.show()

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
if os.path.exists(video_path): os.remove(video_path)
